In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

d_model = 512
num_layers = 6
num_heads = 8
d_k = d_model // num_heads
d_ff = 2048
drop_out_rate = 0.1

src_vocab_size = 10000
trg_vocab_size = 10000

batch_size = 64
seq_len = 128

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

In [2]:
# 토큰 임베딩에 위치 정보(순서) 더해주는 Positional Encoding 모듈
class PositionalEncoding(nn.Module):
    pass

# 소스 입력 인코딩해 문맥 표현 만드는 인코더
class Encoder(nn.Module):
    pass

# 타겟 입력과 encoder outputs 이용해 디코딩 출력 시퀀스 생성하는 디코더
class Decoder(nn.Module):
    pass

In [3]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, trg_vocab_size, d_model):
        super().__init__()
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.trg_embedding = nn.Embedding(trg_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding()
        self.encoder = Encoder()
        self.decoder = Decoder()
        self.output_layer = nn.Linear(d_model, trg_vocab_size)
        self.softmax = nn.LogSoftmax(dim=-1)

    def forward(self, src_inputs, trg_inputs, e_mask=None, d_mask=None):
        src_inputs = self.src_embedding(src_inputs)
        src_inputs = self.positional_encoding(src_inputs)

        trg_inputs = self.trg_embedding(trg_inputs)
        trg_inputs = self.positional_encoding(trg_inputs)

        encoder_outputs = self.encoder(src_inputs, e_mask)
        decoder_outputs = self.decoder(trg_inputs, encoder_outputs, e_mask, d_mask)

        outputs = self.output_layer(decoder_outputs)
        outputs = self.softmax(outputs)

        return outputs

In [4]:
import math

class PositionalEncoding(nn.Module):
    def __init__(self, seq_len, d_model):
        super().__init__()
        pos_encoding = torch.zeros(seq_len, d_model)

        for pos in range(seq_len):
            for i in range(d_model):
                if i % 2 == 0:
                    pos_encoding[pos, i] = math.sin(pos / (10000 ** (2*i / d_model)))
                else:
                    pos_encoding[pos, i] = math.cos(pos / (10000 ** (2*i / d_model)))

        pos_encoding = pos_encoding.unsqueeze(0)
        self.pos_encoding = pos_encoding.to(device).requires_grad_(False)

    def forward(self, x):
        x = x * math.sqrt(d_model)
        x = x + self.pos_encoding
        return x

In [5]:
class FeedForwardLayer(nn.Module):
    def __init__(self, d_model, d_ff, drop_out_rate):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(drop_out_rate)

    def forward(self, x):
        x = self.linear1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x

In [6]:
class LayerNormalization(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.layer = nn.LayerNorm([d_model], elementwise_affine=True, eps=eps)

    def forward(self, x):
        return self.layer(x)

In [7]:
class MultiheadAttention(nn.Module):
    def __init__(self, d_model, dropout_rate):
        super().__init__()

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout_rate)
        self.attn_softmax = nn.Softmax(dim=-1)

        self.w_o = nn.Linear(d_model, d_model)

    def attention(self, q, k, v):
        attn_scores = torch.matmul(q, k.transpose(-1, -2))
        attn_scores /= math.sqrt(d_k)

        attn_weights = self.attn_softmax(attn_scores)
        attn_weights = self.dropout(attn_weights)

        output = torch.matmul(attn_weights, v)
        return output

    def forward(self, q, k, v, mask=None):
        B, T_q, _ = q.size()
        _, T_k, _ = k.size()

        q = self.w_q(q)
        k = self.w_k(k)
        v = self.w_v(v)

        q_heads = q.view(B, T_q, num_heads, d_k).transpose(1, 2)
        k_heads = k.view(B, T_q, num_heads, d_k).transpose(1, 2)
        v_heads = v.view(B, T_q, num_heads, d_k).transpose(1, 2)

        attn_value = self.attention(q_heads, k_heads, v_heads)

        output = attn_value.transpose(1, 2)
        output = output.contiguous().view(B, T_q, d_model)
        output = self.w_o(output)

        return output

In [8]:
class EncoderLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_norm = LayerNormalization(d_model)
        self.multihead_attention = MultiheadAttention(d_model, drop_out_rate)
        self.drop_out = nn.Dropout(drop_out_rate)

        self.layer_norm2 = LayerNormalization(d_model)
        self.feed_forward = FeedForwardLayer(d_model, d_ff, drop_out_rate)
        self.drop_out2 = nn.Dropout(drop_out_rate)

    def forward(self, x, e_mask=None):
        x_1 = self.layer_norm(x)
        x = x + self.drop_out(
            self.multihead_attention(x_1, x_1, x_1, mask=e_mask)
        )
        
        x_2 = self.layer_norm(x)
        x = x + self.drop_out2(
            self.feed_forward(x_2)
        )

        return x

class Encoder(nn.Module):
    def __init__(self, d_model, num_layers):
        super().__init__()
        self.layers = nn.ModuleList([EncoderLayer() for _ in range(num_layers)])
        self.layer_norm = LayerNormalization(d_model)

    def forward(self, x, e_mask):
        for layer in self.layers:
            x = layer(x, e_mask)

        return self.layer_norm(x)

In [9]:
class DecoderLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_norm1 = LayerNormalization(d_model)
        self.masked_multihead_self_attention = MultiheadAttention(d_model, drop_out_rate)
        self.drop_out1 = nn.Dropout(drop_out_rate)
        
        self.layer_norm2 = LayerNormalization(d_model)
        self.multihead_cross_attention = MultiheadAttention(d_model, drop_out_rate)
        self.drop_out2 = nn.Dropout(drop_out_rate)
                                    
        self.layer_norm3 = LayerNormalization(d_model)
        self.feed_forward = FeedForwardLayer(d_model, d_ff, drop_out_rate)
        self.drop_out3 = nn.Dropout(drop_out_rate)

    def forward(self, x, e_outputs, e_mask, d_mask):
        x_1 = self.layer_norm1(x)
        x = x + self.drop_out1(
            self.masked_multihead_self_attention(x_1, x_1, x_1, mask=d_mask)
        )
        
        x_2 = self.layer_norm2(x)
        x = x + self.drop_out2(
            self.multihead_cross_attention(x_2, e_outputs, e_outputs, mask=e_mask)
        )

        x_3 = self.layer_norm3(x)
        x = x + self.drop_out3(
            self.feed_forward(x_3)
        )

        return x

class Decoder(nn.Module):
    def __init__(self, d_model, num_layers):
        super().__init__()
        self.layers = nn.ModuleList([DecoderLayer() for _ in range(num_layers)])
        self.layer_norm = LayerNormalization(d_model)

    def forward(self, x, e_outputs, e_mask, d_mask):
        for layer in self.layers:
            x = layer(x, e_outputs, e_mask, d_mask)

        return self.layer_norm(x)

In [10]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, trg_vocab_size, d_model, seq_len, num_layers):
        super().__init__()
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.trg_embedding = nn.Embedding(trg_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(seq_len, d_model)
        self.encoder = Encoder(d_model, num_layers)
        self.decoder = Decoder(d_model, num_layers)
        self.output_layer = nn.Linear(d_model, trg_vocab_size)
        self.softmax = nn.LogSoftmax(dim=-1)

    def forward(self, src_inputs, trg_inputs, e_mask=None, d_mask=None):
        src_inputs = self.src_embedding(src_inputs)
        src_inputs = self.positional_encoding(src_inputs)

        trg_inputs = self.trg_embedding(trg_inputs)
        trg_inputs = self.positional_encoding(trg_inputs)

        encoder_outputs = self.encoder(src_inputs, e_mask)
        decoder_outputs = self.decoder(trg_inputs, encoder_outputs, e_mask, d_mask)

        outputs = self.output_layer(decoder_outputs)
        outputs = self.softmax(outputs)

        return outputs

model = Transformer(src_vocab_size, trg_vocab_size, d_model, seq_len, num_layers)
model = model.to(device)
print(model)

Transformer(
  (src_embedding): Embedding(10000, 512)
  (trg_embedding): Embedding(10000, 512)
  (positional_encoding): PositionalEncoding()
  (encoder): Encoder(
    (layers): ModuleList(
      (0-5): 6 x EncoderLayer(
        (layer_norm): LayerNormalization(
          (layer): LayerNorm((512,), eps=1e-06, elementwise_affine=True, bias=True)
        )
        (multihead_attention): MultiheadAttention(
          (w_q): Linear(in_features=512, out_features=512, bias=True)
          (w_k): Linear(in_features=512, out_features=512, bias=True)
          (w_v): Linear(in_features=512, out_features=512, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (attn_softmax): Softmax(dim=-1)
          (w_o): Linear(in_features=512, out_features=512, bias=True)
        )
        (drop_out): Dropout(p=0.1, inplace=False)
        (layer_norm2): LayerNormalization(
          (layer): LayerNorm((512,), eps=1e-06, elementwise_affine=True, bias=True)
        )
        (feed_forward):

In [15]:
from torchinfo import summary

summary(model)

Layer (type:depth-idx)                        Param #
Transformer                                   --
├─Embedding: 1-1                              5,120,000
├─Embedding: 1-2                              5,120,000
├─PositionalEncoding: 1-3                     --
├─Encoder: 1-4                                --
│    └─ModuleList: 2-1                        --
│    │    └─EncoderLayer: 3-1                 3,152,384
│    │    └─EncoderLayer: 3-2                 3,152,384
│    │    └─EncoderLayer: 3-3                 3,152,384
│    │    └─EncoderLayer: 3-4                 3,152,384
│    │    └─EncoderLayer: 3-5                 3,152,384
│    │    └─EncoderLayer: 3-6                 3,152,384
│    └─LayerNormalization: 2-2                --
│    │    └─LayerNorm: 3-7                    1,024
├─Decoder: 1-5                                --
│    └─ModuleList: 2-3                        --
│    │    └─DecoderLayer: 3-8                 4,204,032
│    │    └─DecoderLayer: 3-9                 4

In [20]:
src_inputs = torch.randint(0, src_vocab_size, (batch_size, seq_len)).to(device)
trg_inputs = torch.randint(0, trg_vocab_size, (batch_size, seq_len)).to(device)

model.eval()
with torch.no_grad():
    output = model(src_inputs, trg_inputs)

print(f'src Input: {src_inputs.shape}')
print(f'trg Input: {trg_inputs.shape}')
print(f'Model Output: {output.shape}')

src Input: torch.Size([64, 128])
trg Input: torch.Size([64, 128])
Model Output: torch.Size([64, 128, 10000])


## Transformer 정리

Transformer는 Attention을 기반으로 문장의 토큰 간 관계를 학습하는 신경망 구조이다.

### Transformer의 주요 구성 요소

- **Embedding** : 토큰을 벡터로 변환
- **Positional Encoding** : 토큰의 위치와 순서 정보 추가
- **Self-Attention** : 문장 내 다른 토큰과의 관계 계산
- **Multi-Head Attention** : 여러 관점에서 토큰 간 관계를 학습
- **Feed Forward Network** : Attention 결과를 비선형 변환
- **Encoder** : 입력 문장의 의미와 문맥을 표현
- **Decoder** : 이전 출력 토큰을 참고하여 다음 토큰을 생성

### Transformer 모델 구조

| 구조 | 특징 | 대표 모델 | 주요 활용 |
| --- | --- | --- | --- |
| Encoder-only | 입력 문맥 이해에 집중 | BERT, KoELECTRA | 분류, 문장 이해, 임베딩 |
| Decoder-only | 다음 토큰을 반복적으로 예측 | GPT, Qwen, Llama | 텍스트 생성, LLM |
| Encoder-Decoder | 입력을 이해한 뒤 새로운 시퀀스 생성 | T5, NLLB | 번역, 요약, Seq2Seq |

### Decoder 기반 텍스트 생성

Decoder 기반 언어 모델은 이전에 등장한 토큰들을 이용하여 다음 토큰을 예측한다.

```text
입력 문장
    ↓
Tokenizer
    ↓
Transformer Decoder
    ↓
다음 토큰 확률 계산
    ↓
다음 토큰 선택
    ↓
생성된 토큰을 입력에 추가
    ↓
반복
    ↓
문장 생성
```